# 第 5 课：LangGraph 状态机工作流

预计用时：90 分钟  
适合人群：完成上一课的零基础学习者；本 Notebook 也包含独立运行所需的准备代码。

## 学习目标

- 把复杂任务拆成节点与边
- 用结构化 State 传递数据
- 实现条件分支、重试和 checkpoint

## 学习方式

按顺序运行每个代码单元格。先阅读预测结果，再运行验证；遇到报错先看本课“常见问题”，不要直接跳过。带有真实模型或外网请求的示例默认注释，确认 API Key 与费用后再启用。


## 1. 先理解概念

图工作流适合步骤固定但存在分支与重试的任务。节点读取状态并只返回需要更新的字段；边决定下一步。显式状态让流程比隐藏在提示词里的规划更容易测试与恢复。

### 本课路线

1. 定义 `TravelState`
2. 逐个实现解析、搜索、天气和估价节点
3. 实现预算条件路由
4. 组装并编译图
5. 使用 `thread_id` 运行并保存状态


## 2. 运行前检查

1. 从项目根目录启动 Jupyter Lab。
2. 选择项目 `.venv` 对应的 Python 内核。
3. 若本课调用百炼，先在启动 Jupyter 的终端设置 `DASHSCOPE_API_KEY`。
4. 不要把 Key 粘贴到单元格、截图或 Git 提交中。

> 下方“准备代码”可能与前课重复，这是为了保证每个 Notebook 都能单独运行。初学时建议展开阅读，熟悉后可折叠。


### 准备代码


In [ ]:
# %pip install -q openai pydantic>=2.7 httpx fastapi uvicorn fastmcp langgraph langfuse ragas numpy pytest

import os
from dotenv import load_dotenv

load_dotenv()

# 推荐在启动 Jupyter 前设置：
# Windows PowerShell: $env:DASHSCOPE_API_KEY='sk-...'
# macOS/Linux:       export DASHSCOPE_API_KEY='sk-...'

BAILIAN_API_KEY = os.getenv('DASHSCOPE_API_KEY', '')
BAILIAN_BASE_URL = os.getenv(
    'BAILIAN_BASE_URL',
    'https://dashscope.aliyuncs.com/compatible-mode/v1',
)
BAILIAN_MODEL = os.getenv('BAILIAN_MODEL', 'qwen-plus')
BAILIAN_EMBEDDING_MODEL = os.getenv('BAILIAN_EMBEDDING_MODEL', 'text-embedding-v4')

print('模型:', BAILIAN_MODEL)
print('Base URL:', BAILIAN_BASE_URL)
print('API Key:', '已配置' if BAILIAN_API_KEY else '未配置（调用模型前必须设置）')


### 准备代码


In [ ]:
from __future__ import annotations

import asyncio
import json
import logging
import math
import sqlite3
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Awaitable, Callable, Literal, TypedDict

import httpx
import numpy as np
from openai import AsyncOpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

WORKSPACE = (Path.cwd() / 'agent_workspace').resolve()
WORKSPACE.mkdir(exist_ok=True)

def require_api_key() -> None:
    if not BAILIAN_API_KEY:
        raise RuntimeError('请先设置环境变量 DASHSCOPE_API_KEY，然后重新运行配置单元格。')

client = AsyncOpenAI(api_key=BAILIAN_API_KEY or 'missing', base_url=BAILIAN_BASE_URL)
print('工作目录:', WORKSPACE)


### 准备代码


In [ ]:
class AgentLimits(BaseModel):
    model_config = ConfigDict(extra='forbid')
    max_steps: int = Field(default=8, ge=1, le=30)
    model_timeout_s: float = Field(default=45, gt=0, le=300)
    tool_timeout_s: float = Field(default=15, gt=0, le=120)
    total_timeout_s: float = Field(default=120, gt=0, le=600)

class ToolResult(BaseModel):
    ok: bool
    data: Any = None
    error: str | None = None
    retryable: bool = False

ToolHandler = Callable[[BaseModel], Awaitable[Any]]

@dataclass
class RegisteredTool:
    name: str
    description: str
    args_model: type[BaseModel]
    handler: ToolHandler
    side_effect: bool = False

    def openai_schema(self) -> dict[str, Any]:
        schema = self.args_model.model_json_schema()
        schema['additionalProperties'] = False
        return {
            'type': 'function',
            'function': {
                'name': self.name,
                'description': self.description,
                'parameters': schema,
            },
        }

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, RegisteredTool] = {}

    def register(self, tool: RegisteredTool) -> None:
        if tool.name in self._tools:
            raise ValueError(f'工具重复注册: {tool.name}')
        self._tools[tool.name] = tool

    @property
    def schemas(self) -> list[dict[str, Any]]:
        return [tool.openai_schema() for tool in self._tools.values()]

    async def execute(self, name: str, raw_arguments: str, timeout_s: float) -> ToolResult:
        tool = self._tools.get(name)
        if tool is None:
            return ToolResult(ok=False, error=f'未知工具: {name}', retryable=False)
        try:
            arguments = json.loads(raw_arguments or '{}')
            validated = tool.args_model.model_validate(arguments)
        except json.JSONDecodeError as exc:
            return ToolResult(ok=False, error=f'工具参数不是合法 JSON: {exc}')
        except ValidationError as exc:
            return ToolResult(ok=False, error=f'工具参数校验失败: {exc}')
        try:
            async with asyncio.timeout(timeout_s):
                value = await tool.handler(validated)
            return ToolResult(ok=True, data=value)
        except TimeoutError:
            return ToolResult(ok=False, error=f'工具 {name} 执行超时', retryable=True)
        except httpx.HTTPStatusError as exc:
            retryable = exc.response.status_code in {408, 429, 500, 502, 503, 504}
            return ToolResult(ok=False, error=f'上游 HTTP {exc.response.status_code}', retryable=retryable)
        except Exception as exc:
            return ToolResult(ok=False, error=f'{type(exc).__name__}: {exc}', retryable=False)


### 准备代码


In [ ]:
class WeatherArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    city: str = Field(min_length=1, max_length=80)

class ExchangeArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    amount: float = Field(gt=0, le=10_000_000)
    from_currency: str = Field(pattern=r'^[A-Za-z]{3}$')
    to_currency: str = Field(pattern=r'^[A-Za-z]{3}$')

class TodoArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    action: Literal['add', 'list', 'complete', 'delete']
    title: str | None = Field(default=None, max_length=200)
    todo_id: int | None = Field(default=None, ge=1)

class LogArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    level: Literal['INFO', 'WARNING', 'ERROR'] = 'INFO'
    message: str = Field(min_length=1, max_length=1000)
    metadata: dict[str, Any] = Field(default_factory=dict)

class SearchArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    query: str = Field(min_length=2, max_length=200)
    max_results: int = Field(default=5, ge=1, le=10)

async def get_weather(args: WeatherArgs) -> dict[str, Any]:
    async with httpx.AsyncClient(timeout=10) as http:
        geo = await http.get('https://geocoding-api.open-meteo.com/v1/search', params={
            'name': args.city, 'count': 1, 'language': 'zh', 'format': 'json'
        })
        geo.raise_for_status()
        results = geo.json().get('results') or []
        if not results:
            return {'found': False, 'city': args.city}
        place = results[0]
        weather = await http.get('https://api.open-meteo.com/v1/forecast', params={
            'latitude': place['latitude'],
            'longitude': place['longitude'],
            'current': 'temperature_2m,apparent_temperature,precipitation,weather_code',
            'timezone': 'auto',
        })
        weather.raise_for_status()
        return {'found': True, 'city': place['name'], 'country': place.get('country'), **weather.json()['current']}

async def convert_currency(args: ExchangeArgs) -> dict[str, Any]:
    source, target = args.from_currency.upper(), args.to_currency.upper()
    if source == target:
        return {'amount': args.amount, 'from': source, 'to': target, 'converted': args.amount, 'rate': 1}
    async with httpx.AsyncClient(timeout=10) as http:
        response = await http.get('https://api.frankfurter.app/latest', params={'amount': args.amount, 'from': source, 'to': target})
        response.raise_for_status()
        data = response.json()
        converted = data['rates'][target]
        return {'amount': args.amount, 'from': source, 'to': target, 'converted': converted, 'rate': converted / args.amount}

TODO_DB = WORKSPACE / 'todos.sqlite3'

def init_todo_db() -> None:
    with sqlite3.connect(TODO_DB) as conn:
        conn.execute('CREATE TABLE IF NOT EXISTS todos (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)')

async def manage_todo(args: TodoArgs) -> list[dict[str, Any]] | dict[str, Any]:
    init_todo_db()
    with sqlite3.connect(TODO_DB) as conn:
        conn.row_factory = sqlite3.Row
        if args.action == 'add':
            if not args.title:
                raise ValueError('add 操作必须提供 title')
            cursor = conn.execute('INSERT INTO todos(title) VALUES (?)', (args.title,))
            return {'id': cursor.lastrowid, 'title': args.title, 'done': False}
        if args.action in {'complete', 'delete'} and not args.todo_id:
            raise ValueError(f'{args.action} 操作必须提供 todo_id')
        if args.action == 'complete':
            cursor = conn.execute('UPDATE todos SET done=1 WHERE id=?', (args.todo_id,))
            return {'updated': cursor.rowcount}
        if args.action == 'delete':
            cursor = conn.execute('DELETE FROM todos WHERE id=?', (args.todo_id,))
            return {'deleted': cursor.rowcount}
        rows = conn.execute('SELECT id, title, done FROM todos ORDER BY id').fetchall()
        return [dict(row) for row in rows]

async def write_log(args: LogArgs) -> dict[str, Any]:
    record = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'level': args.level,
        'message': args.message,
        'metadata': args.metadata,
    }
    path = WORKSPACE / 'agent.jsonl'
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    return {'written': True, 'path': str(path)}

async def web_search(args: SearchArgs) -> list[dict[str, str]]:
    async with httpx.AsyncClient(timeout=10, headers={'User-Agent': 'agent-learning-notebook/1.0'}) as http:
        response = await http.get('https://zh.wikipedia.org/w/api.php', params={
            'action': 'query', 'list': 'search', 'srsearch': args.query,
            'format': 'json', 'utf8': 1, 'srlimit': args.max_results,
        })
        response.raise_for_status()
        return [
            {'title': item['title'], 'snippet': item['snippet'], 'url': f"https://zh.wikipedia.org/wiki/{item['title'].replace(' ', '_')}"}
            for item in response.json()['query']['search']
        ]

registry = ToolRegistry()
for tool in [
    RegisteredTool('get_weather', '查询城市当前天气', WeatherArgs, get_weather),
    RegisteredTool('convert_currency', '按最新公开汇率换算货币', ExchangeArgs, convert_currency),
    RegisteredTool('manage_todo', '添加、列出、完成或删除待办', TodoArgs, manage_todo, side_effect=True),
    RegisteredTool('write_log', '写入一条结构化日志', LogArgs, write_log, side_effect=True),
    RegisteredTool('web_search', '搜索百科资料，返回标题、摘要和链接', SearchArgs, web_search),
]:
    registry.register(tool)

agent = MinimalAgent(registry)
print([schema['function']['name'] for schema in registry.schemas])


### 核心实验


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

class TravelState(TypedDict, total=False):
    request: str
    origin: str
    budget: float
    candidates: list[str]
    weather: dict[str, Any]
    estimated_costs: dict[str, float]
    selected: str
    attempts: int
    itinerary: str
    errors: list[str]

async def parse_request(state: TravelState) -> dict[str, Any]:
    # 教学版先使用显式默认值；练习：让百炼输出严格 JSON，再用 Pydantic 验证。
    return {'origin': state.get('origin', '杭州'), 'budget': state.get('budget', 1500), 'attempts': 0, 'errors': []}

async def find_destinations(state: TravelState) -> dict[str, Any]:
    candidates = ['苏州', '南京', '宁波'] if state['origin'] == '杭州' else ['杭州', '苏州', '南京']
    return {'candidates': candidates}

async def check_weather_node(state: TravelState) -> dict[str, Any]:
    calls = [get_weather(WeatherArgs(city=city)) for city in state['candidates']]
    results = await asyncio.gather(*calls, return_exceptions=True)
    weather = {city: result for city, result in zip(state['candidates'], results) if not isinstance(result, Exception)}
    errors = list(state.get('errors', []))
    errors.extend(str(result) for result in results if isinstance(result, Exception))
    return {'weather': weather, 'errors': errors}

def estimate_prices(state: TravelState) -> dict[str, Any]:
    # 可替换为真实交通/酒店工具。固定值使测试可重复。
    base = {'苏州': 900.0, '南京': 1250.0, '宁波': 1100.0, '杭州': 1000.0}
    return {'estimated_costs': {city: base.get(city, 1400.0) for city in state['candidates']}}

def choose_destination(state: TravelState) -> dict[str, Any]:
    affordable = [(cost, city) for city, cost in state['estimated_costs'].items() if cost <= state['budget']]
    if not affordable:
        return {'selected': '', 'attempts': state.get('attempts', 0) + 1}
    return {'selected': min(affordable)[1]}

def route_after_choice(state: TravelState) -> Literal['plan', 'retry', 'fail']:
    if state.get('selected'):
        return 'plan'
    return 'retry' if state.get('attempts', 0) < 2 else 'fail'

def relax_search(state: TravelState) -> dict[str, Any]:
    return {'budget': state['budget'] * 1.15}

def build_itinerary(state: TravelState) -> dict[str, Any]:
    city = state['selected']
    cost = state['estimated_costs'][city]
    return {'itinerary': f'周六上午从{state["origin"]}出发前往{city}；下午城市漫步；周日文化景点和当地美食；预计 ¥{cost:.0f}。'}

def fail_plan(state: TravelState) -> dict[str, Any]:
    return {'itinerary': '在当前预算和候选范围内没有可行方案，请调整预算或目的地范围。'}

builder = StateGraph(TravelState)
for name, node in [
    ('parse', parse_request), ('search', find_destinations), ('weather', check_weather_node),
    ('prices', estimate_prices), ('choose', choose_destination), ('relax', relax_search),
    ('plan', build_itinerary), ('fail', fail_plan),
]:
    builder.add_node(name, node)
builder.add_edge(START, 'parse')
builder.add_edge('parse', 'search')
builder.add_edge('search', 'weather')
builder.add_edge('weather', 'prices')
builder.add_edge('prices', 'choose')
builder.add_conditional_edges('choose', route_after_choice, {'plan': 'plan', 'retry': 'relax', 'fail': 'fail'})
builder.add_edge('relax', 'prices')
builder.add_edge('plan', END)
builder.add_edge('fail', END)
travel_graph = builder.compile(checkpointer=InMemorySaver())

# result = await travel_graph.ainvoke(
#     {'request': '从杭州出发，预算 1500 元规划周末旅行', 'origin': '杭州', 'budget': 1500},
#     {'configurable': {'thread_id': 'demo-user-1'}},
# )
# print(result['itinerary'])


## 3. 观察与验证

核心代码中的真实 API 调用默认被注释。先运行无需额度的断言或定义单元格；确认输出和预期一致后，再逐行取消示例注释。


## 4. 代码讲解

为了让本课独立运行，准备代码也包含天气工具。固定价格让测试可重复；天气是外部依赖，失败会被记录到 `errors` 而不会让整个图立刻崩溃。

调试建议：从报错的最后一行开始读，确认当前 Notebook 的单元格是否按顺序全部运行；若看到 `NameError`，通常是准备单元格未运行或内核已重启。


## 5. 常见问题

- **`ModuleNotFoundError`**：确认选中了 `.venv` 内核，并重新安装 `requirements.txt`。
- **提示未配置 API Key**：在启动 Jupyter 的同一个终端设置环境变量，然后重启内核。
- **网络超时或 429**：公开接口或模型服务可能限流；稍后重试，不要移除超时保护。
- **运行结果和预期不同**：先执行“Restart Kernel and Run All”，排除旧变量残留。
- **产生费用吗？**：只有实际调用百炼聊天或 Embedding 接口才会消耗额度；本地定义、SQLite 和断言不会。

## 6. 练习

- 把预算调低到触发 relax 分支
- 加入第四个候选城市
- 让天气不佳的城市获得一个惩罚分

建议先复制相关单元格再修改，保留一份能工作的基线。


## 7. 本课验收

完成后逐项确认：

- [ ] 能画出图的节点和条件边
- [ ] 重试不会无限循环
- [ ] 相同 `thread_id` 可关联同一会话

如果某项还解释不清，回到对应代码，用更小的输入单独调用函数，而不是直接运行完整 Agent。


## 下一步

继续学习 `06_三层记忆.ipynb`。

> 学习记录建议：写下今天最重要的一个概念、遇到的一个错误、以及你如何验证修复。
